In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

In [2]:
dataset = pd.read_csv(r'../data/NepalLatestAQI.csv')
dataset.head()

,date,station,latitude,longitude,aqi,pm2_5,pm10,no2,so2,o3,temperature_C,relative_humidity_%,notes
0,2024-01-01,Kathmandu,27.7172,85.3240,162,75.9,161.5,18.5,8.0,16.2,20.9,65.2,NaN
1,2024-01-01,Lalitpur,27.6648,85.3188,157,66.9,105.9,18.2,4.2,34.1,28.8,76.2,NaN
2,2024-01-01,Bhaktapur,27.6714,85.4270,166,84.8,182.9,16.4,7.1,26.7,20.3,60.9,NaN
3,2024-01-01,Pokhara,28.2096,83.9856,79,25.5,47.8,2.0,1.0,31.7,23.0,47.9,NaN
4,2024-01-01,Biratnagar,26.4525,87.2718,146,53.8,86.3,17.2,1.9,34.5,23.9,54.1,NaN


In [3]:
dataset.isnull().sum()

date                      0
station                   0
latitude                  0
longitude                 0
aqi                       0
pm2_5                     0
pm10                      0
no2                       0
so2                       0
o3                        0
temperature_C             0
relative_humidity_%       0
notes                  9761
dtype: int64

In [4]:
dataset = dataset.drop(columns='notes')

In [5]:
dataset = dataset.rename(columns=lambda c: c.strip())

In [6]:
dataset = dataset.sort_values('date').drop_duplicates(subset=['station', 'date'], keep='last')

In [7]:
dataset['date'] = pd.to_datetime(dataset['date'])
dataset['month'] = dataset['date'].dt.month
dataset['day'] = dataset['date'].dt.day
dataset['dayofweek'] = dataset['date'].dt.dayofweek
dataset['is_weekend'] = dataset['dayofweek'].isin([5,6]).astype(int)

❗ Why do we encode cyclical features?

Months and weekdays are cyclical:

After December (12) comes January (1)

After Sunday (6) comes Monday (0)

If you use raw numbers:

The model thinks January (1) is far from December (12)
But they are next to each other.


In [8]:
dataset['month_sin'] = np.sin(2 * np.pi * dataset['month']/12)
dataset['month_cos'] = np.cos(2 * np.pi * dataset['month']/12)
dataset['dow_sin'] = np.sin(2 * np.pi * dataset['dayofweek']/7)
dataset['dow_cos'] = np.cos(2 * np.pi * dataset['dayofweek']/7)

In [9]:
dataset.columns

Index(['date', 'station', 'latitude', 'longitude', 'aqi', 'pm2_5', 'pm10',
       'no2', 'so2', 'o3', 'temperature_C', 'relative_humidity_%', 'month',
       'day', 'dayofweek', 'is_weekend', 'month_sin', 'month_cos', 'dow_sin',
       'dow_cos'],
      dtype='object')

In [10]:
cols = ['pm2_5', 'pm10', 'no2', 'so2', 'o3', 'temperature_C', 'relative_humidity_%']

In [11]:
lags = [1, 3, 7]
rolls = [3, 7]

In [12]:
dataset = dataset.sort_values(['station', 'date']).reset_index(drop=True)

# Lags & rolling use only past values (shifted), so no leakage occurs


In [13]:
# create lag feature
for c in cols:
    for lag in lags:
        new_col = f"{c}_lag{lag}"
        dataset[new_col] = dataset.groupby('station')[c].shift(lag)

In [14]:
# crete rolling-mean features
for c in cols:
    for w in rolls:
        new_col = f"{c}_roll{w}"
        dataset[new_col] = (dataset.groupby('station')[c].transform(lambda s: s.rolling(window=w, min_periods=1).mean().shift(1)))

In [15]:
new_features = [c for c in dataset.columns if any(x in c for x in ['_lag', '_roll'])]
print(f"Added {len(new_features)} features: {new_features[:20]}{'' if len(new_features)<=20 else ' ...'}")
print("\nMissing values count for new features:")
print(dataset[new_features].isnull().sum())

Added 35 features: ['pm2_5_lag1', 'pm2_5_lag3', 'pm2_5_lag7', 'pm10_lag1', 'pm10_lag3', 'pm10_lag7', 'no2_lag1', 'no2_lag3', 'no2_lag7', 'so2_lag1', 'so2_lag3', 'so2_lag7', 'o3_lag1', 'o3_lag3', 'o3_lag7', 'temperature_C_lag1', 'temperature_C_lag3', 'temperature_C_lag7', 'relative_humidity_%_lag1', 'relative_humidity_%_lag3'] ...

Missing values count for new features:
pm2_5_lag1                    15
pm2_5_lag3                    45
pm2_5_lag7                   105
pm10_lag1                     15
pm10_lag3                     45
pm10_lag7                    105
no2_lag1                      15
no2_lag3                      45
no2_lag7                     105
so2_lag1                      15
so2_lag3                      45
so2_lag7                     105
o3_lag1                       15
o3_lag3                       45
o3_lag7                      105
temperature_C_lag1            15
temperature_C_lag3            45
temperature_C_lag7           105
relative_humidity_%_lag1      15
r

In [16]:
initial_len = len(dataset)
dataset = dataset[~dataset[new_features].isnull().any(axis=1)].reset_index(drop=True)
dropped = initial_len - len(dataset)
print(f"Dropped {dropped} rows ({dropped/initial_len*100:.2f}%) because lag/roll features were not available.")

Dropped 105 rows (1.07%) because lag/roll features were not available.


In [17]:
dataset['station'].nunique()

15

In [18]:
dataset = pd.get_dummies(dataset, columns=['station'], prefix='st', drop_first=False)

In [19]:
dataset.shape

(9735, 69)

In [20]:
train_end = '2025-05-30'
valid_end = '2025-08-30'

In [21]:
train_data = dataset[ dataset['date'] <= train_end ]
valid_data = dataset[ (dataset['date'] > train_end) & (dataset['date'] <= valid_end) ]
test_data = dataset[ dataset['date'] > valid_end ]

In [22]:
print("Train shape :", train_data.shape)
print("Valid shape :", valid_data.shape)
print("Test shape  :", test_data.shape)

Train shape : (7635, 69)
Valid shape : (1380, 69)
Test shape  : (720, 69)


In [23]:
target = 'aqi'
features = [c for c in dataset.columns if c not in ['date', 'aqi']]

In [24]:
x_train = train_data[features]
y_train = train_data[target]

x_valid = valid_data[features]
y_valid = valid_data[target]

x_test = test_data[features]
y_test = test_data[target]

In [25]:
from xgboost import XGBRegressor

In [26]:
model = XGBRegressor( n_estimators= 1844, 
learning_rate= 0.005138423447513474, 
max_depth= 6, 
subsample= 0.6011047426660859, 
colsample_bytree= 0.8897617488503717, 
reg_alpha= 0.0065837707426329595, 
reg_lambda= 8.5800788151842, 
min_child_weight= 2, 
gamma= 1.4098179177734331,
tree_method="hist", 
verbosity=0,
early_stopping_rounds=50)

model.fit(x_train, y_train, eval_set=[(x_valid, y_valid)], verbose=False)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8897617488503717
,device,None
,early_stopping_rounds,50
,enable_categorical,False
,eval_metric,None


In [27]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [28]:
def compute_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / np.clip(y_true, 1e-6, None))) * 100
    return {'mae': mae, 'rmse': rmse, 'mape': mape}

In [29]:
pred_valid = model.predict(x_valid)
pred_test = model.predict(x_test)

In [30]:
metrics_valid = compute_metrics(y_valid, pred_valid)
metrics_test  = compute_metrics(y_test, pred_test)

print("Validation:", metrics_valid)
print("Test      :", metrics_test)

Validation: {'mae': 5.707831859588623, 'rmse': 31.911090850830078, 'mape': np.float64(13.856653205229883)}
Test      : {'mae': 3.11328125, 'rmse': 23.191118240356445, 'mape': np.float64(2.8501186422489297)}


In [31]:
model.score(x_test, y_test)*100

59.24973487854004

In [32]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [33]:
import copy
import pprint

In [34]:
SEED = 42

In [35]:
def create_study(n_trials=50, timeout=None, seed=SEED):
    """Create an Optuna study with a sensible sampler and pruner."""
    sampler = TPESampler(seed=seed)
    pruner = MedianPruner(n_warmup_steps=5)
    study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
    return study

In [36]:
def objective(trial):

    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 3000, step=50),
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "subsample": trial.suggest_float("subsample", 0.4, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 1e-8, 10.0, log=True),
        "tree_method": "hist",
        "random_state": SEED,
        "verbosity": 0,
        "n_jobs": -1,
    }

    model2 = XGBRegressor(**params)

   
    model2.fit(
        x_train, y_train,
        eval_set=[(x_valid, y_valid)],
        verbose=False,
        )
    

    preds = model2.predict(x_valid)
    mae = mean_absolute_error(y_valid, preds)

    return mae


In [37]:
n_trials = 50
study = create_study(n_trials=n_trials)

print("Starting Optuna study with", n_trials, "trials...")
study.optimize(objective, n_trials=n_trials)

print("\nBest study results:")
print("  Best MAE (validation): {:.4f}".format(study.best_value))
print("  Best trial number:", study.best_trial.number)
print("  Best params:")
pprint.pprint(study.best_params)


[I 2025-12-23 19:51:20,581] A new study created in memory with name: no-name-ccc454d4-1949-4301-863a-f29fb2a41446


Starting Optuna study with 50 trials...


[I 2025-12-23 19:51:34,445] Trial 0 finished with value: 10.009233474731445 and parameters: {'n_estimators': 1200, 'learning_rate': 0.20218499516556748, 'max_depth': 8, 'subsample': 0.759195090518222, 'colsample_bytree': 0.4936111842654619, 'reg_alpha': 2.5348407664333426e-07, 'reg_lambda': 3.3323645788192616e-08, 'min_child_weight': 9, 'gamma': 0.002570603566117598}. Best is trial 0 with value: 10.009233474731445.
[I 2025-12-23 19:54:45,101] Trial 1 finished with value: 36.742191314697266 and parameters: {'n_estimators': 2150, 'learning_rate': 0.00011791655502618509, 'max_depth': 10, 'subsample': 0.899465584480253, 'colsample_bytree': 0.5274034664069657, 'reg_alpha': 4.329370014459266e-07, 'reg_lambda': 4.4734294104626844e-07, 'min_child_weight': 4, 'gamma': 0.00052821153945323}. Best is trial 0 with value: 10.009233474731445.
[I 2025-12-23 19:55:22,925] Trial 2 finished with value: 15.049131393432617 and parameters: {'n_estimators': 1350, 'learning_rate': 0.001029530064265006, 'max_d


Best study results:
  Best MAE (validation): 5.4950
  Best trial number: 25
  Best params:
{'colsample_bytree': 0.8368777908536602,
 'gamma': 0.5838007844449354,
 'learning_rate': 0.019831145240939582,
 'max_depth': 8,
 'min_child_weight': 2,
 'n_estimators': 950,
 'reg_alpha': 0.2901995076939732,
 'reg_lambda': 0.8431493068165524,
 'subsample': 0.9333221943540619}
